# AutoLab — étape 8 : première boucle complète sur Colab (GPU T4)

Muse Spark (Meta) propose des substitutions, le code fabrique les cristaux, UMA les relaxe sur le GPU, les filtres les trient (E_hull UMA, band gap ALIGNN, liaisons S–S), puis Muse Spark analyse les résultats et oriente l'itération suivante. Baseten (Muse Spark Open) sert de secours si Meta échoue.

**Boucle minimale** : 2 itérations, 5 candidats au plus par itération, sans électrodes ni dynamique moléculaire, 250 nouvelles relaxations UMA au plus.
**Budget des modèles de langage** (plafonds stricts dans le code) : Meta 1 $, Baseten 0,50 $.

**Avant de lancer**
1. *Exécution → Modifier le type d'exécution →* **GPU T4**.
2. Panneau **Secrets** (icône clé) : `MP_API_KEY`, `HF_TOKEN`, `MODEL_API_KEY`, `BASETEN_API_KEY`, avec **Accès au notebook** activé. Les valeurs ne sont jamais affichées.
3. Panneau **Fichiers** (icône dossier) : déposer `autolab_seed.zip` (sur le PC : `D:\autolab\.cache\autolab_seed.zip`) dans `/content/`. Sinon, la cellule « cache de départ » ouvrira une fenêtre pour le choisir.
4. *Exécution → Tout exécuter.* Durée prévue : 45 à 75 minutes.

Chaque cellule capture ses erreurs : le notebook va jusqu'au bout même si une étape échoue. À la fin : un rapport unique et `autolab_loop_results.zip` à télécharger.

In [ ]:
# Helpers: every step records its status and duration; errors never stop the notebook.
import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

REPO_URL = "https://github.com/hkdebendo/autolab.git"
REPO = Path("/content/autolab")
WORK = Path("/content/autolab_run")
WORK.mkdir(parents=True, exist_ok=True)
CACHE = REPO / ".cache"
LOOP_OUT = WORK / "loop"
SEED = Path("/content/autolab_seed.zip")
MAX_RELAXATIONS = 250
T_START = time.time()
REPORT = {"steps": {}, "results": {}}
# Subprocesses get a clean interpreter (fresh imports after pip install), the local UMA backend and Meta as brain.
ENV = dict(os.environ, SIM_BACKEND="local", LLM_PROVIDER="meta", PYTHONUNBUFFERED="1", AUTOLAB_CACHE_DIR=str(CACHE))


def save_report():
    (WORK / "report.json").write_text(json.dumps(REPORT, indent=2, default=str), encoding="utf-8")


def record(step, status, seconds, **info):
    REPORT["steps"][step] = {"status": status, "seconds": round(seconds, 1), **info}
    save_report()
    print(f"\n[{step}] {status} en {seconds / 60:.1f} min")


def sh(step, cmd, cwd=REPO):
    """Run a command, stream its output live and into WORK/<step>.log; never raises."""
    t0 = time.time()
    try:
        with open(WORK / f"{step}.log", "w", encoding="utf-8") as log:
            proc = subprocess.Popen([str(c) for c in cmd], cwd=cwd, env=ENV, text=True,
                                    stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
            try:
                for line in proc.stdout:
                    print(line, end="")
                    log.write(line)
                code = proc.wait()
            except KeyboardInterrupt:
                proc.kill()
                record(step, "interrompu", time.time() - t0)
                return False
        record(step, "ok" if code == 0 else f"échec (code {code})", time.time() - t0)
        return code == 0
    except Exception as e:
        record(step, f"erreur : {type(e).__name__}: {e}", time.time() - t0)
        return False


def py(step, source):
    """Run a Python snippet in a fresh process from the repository folder."""
    script = WORK / f"{step}.py"
    script.write_text(source, encoding="utf-8")
    return sh(step, [sys.executable, script])


def load(name):
    p = WORK / f"{name}.json"
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else None


print("Helpers prêts.")

In [ ]:
# GPU check
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip() or "aucun GPU"
except Exception as e:
    gpu = f"aucun GPU ({type(e).__name__})"
REPORT["gpu"] = gpu
REPORT["python"] = sys.version.split()[0]
save_report()
print("GPU :", gpu, "| Python", REPORT["python"])
if "T4" not in gpu:
    print("⚠️ Pas de T4 : changer le type d'exécution (GPU T4).")

In [ ]:
# Keys from Colab Secrets: only their presence is ever printed.
REPORT["secrets"] = {}
try:
    from google.colab import userdata
except Exception:
    userdata = None
for name in ("MP_API_KEY", "HF_TOKEN", "MODEL_API_KEY", "BASETEN_API_KEY"):
    present, why = False, ""
    try:
        value = userdata.get(name) if userdata else os.environ.get(name, "")
        if value and value.strip():
            ENV[name] = value.strip()
            present = True
    except Exception as e:
        why = f" ({type(e).__name__} : secret absent ou accès au notebook non activé)"
    value = None
    REPORT["secrets"][name] = "présent" if present else "absent"
    print(f"{name} : {'présent' if present else 'ABSENT' + why}")
save_report()

In [ ]:
# Phase A: clone (or update) the public repository and install it with ALIGNN and fairchem.
if (REPO / ".git").exists():
    ok = sh("A_clone", ["git", "-C", REPO, "pull", "--ff-only"], cwd="/content")
else:
    ok = sh("A_clone", ["git", "clone", "--depth", "1", REPO_URL, REPO], cwd="/content")
if ok:
    sh("A_install", [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[bandgap,dev]",
                     "fairchem-core==2.24.0"])
try:
    REPORT["commit"] = subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"],
                                      capture_output=True, text=True).stdout.strip()
except Exception as e:
    REPORT["commit"] = f"inconnu ({type(e).__name__})"
py("A_versions", r"""
import json, torch
from importlib.metadata import version
out = {p: version(p) for p in ("fairchem-core", "alignn", "torch", "pymatgen", "openai")}
out["cuda"] = torch.cuda.is_available()
open("/content/autolab_run/A_versions.json", "w").write(json.dumps(out))
print(out)
""")
REPORT["versions"] = load("A_versions")
save_report()
print("Commit testé :", REPORT["commit"])

In [ ]:
# Seed cache: the 54 UMA relaxations of step 6 (52 Li-P-S references + LiCl + LiS) and the MP references.
t0 = time.time()
try:
    if not SEED.exists():
        from google.colab import files
        print("Choisir autolab_seed.zip (D:\\autolab\\.cache\\autolab_seed.zip sur le PC) :")
        for name, data in files.upload().items():
            SEED.write_bytes(data)
    with zipfile.ZipFile(SEED) as z:
        z.extractall(CACHE)
    n = len(list((CACHE / "relax").glob("*.json")))
    record("B_seed", "ok", time.time() - t0, relaxations_in_cache=n)
    print("Relaxations en cache :", n, "(attendu : 54)")
except Exception as e:
    record("B_seed", f"erreur : {type(e).__name__}: {e}", time.time() - t0)

In [ ]:
# Check that the 52 Li-P-S references are reused from the cache on this machine (max_new=0: no computation).
py("B_reuse", r"""
import json
from autolab.data.mp import get_references
from autolab.sim.backend import BudgetExceeded, relax_structures
refs = get_references(["Li", "P", "S"])
try:
    relax_structures([r["structure"] for r in refs], max_new=0)
    out = {"references": len(refs), "reused": len(refs)}
except BudgetExceeded as e:
    out = {"references": len(refs), "reused": None, "error": str(e)}
open("/content/autolab_run/B_reuse.json", "w").write(json.dumps(out))
print(out)
""")
REPORT["results"]["reuse"] = load("B_reuse")
save_report()

In [ ]:
# ALIGNN models: downloaded once here (Figshare), checked on LiCl (expected optB88 gap > 5.5 eV).
py("C_alignn", r"""
import json, time
from pymatgen.core import Lattice, Structure
from autolab.analysis.bandgap import evaluate_bandgap
t0 = time.perf_counter()
licl = Structure.from_spacegroup("Fm-3m", Lattice.cubic(5.13), ["Li", "Cl"], [[0, 0, 0], [0.5, 0.5, 0.5]])
out = {**evaluate_bandgap(licl), "seconds": round(time.perf_counter() - t0, 1)}
open("/content/autolab_run/C_alignn.json", "w").write(json.dumps(out))
print(out)
""")
REPORT["results"]["alignn"] = load("C_alignn")
save_report()

In [ ]:
# The loop: 2 iterations, <= 5 candidates each, <= 250 new UMA relaxations, Meta <= $1, Baseten <= $0.50.
sh("D_loop", [sys.executable, "scripts/run_loop.py", "--iterations", 2, "--candidates", 5,
              "--max-relaxations", MAX_RELAXATIONS, "--brain", "meta", "--fallback", "baseten",
              "--brain-budget", 1.0, "--fallback-budget", 0.5, "--out", LOOP_OUT])

In [ ]:
# Final report (copy everything below) + zip of the cache and results to download.
lines = ["# Rapport étape 8 — première boucle AutoLab sur Colab",
         f"GPU : {REPORT.get('gpu')} | Python {REPORT.get('python')} | commit {REPORT.get('commit')}",
         f"Versions : {REPORT.get('versions')}", f"Secrets : {REPORT.get('secrets')}", ""]
try:
    reuse = REPORT["results"].get("reuse") or {}
    alignn = REPORT["results"].get("alignn") or {}
    lines.append(f"Cache de départ : {REPORT['steps'].get('B_seed', {}).get('relaxations_in_cache')} relaxations ; "
                 f"références Li-P-S réutilisées : {reuse.get('reused')}/{reuse.get('references')} "
                 f"{reuse.get('error', '')}")
    lines.append(f"ALIGNN sur LiCl : optB88 {alignn.get('gap_optb88')} eV (attendu > 5,5), "
                 f"mBJ {alignn.get('gap_mbj')} eV, {alignn.get('seconds')} s")
    lines.append("")
    report_md = LOOP_OUT / "report.md"
    if report_md.exists():
        lines.append(report_md.read_text(encoding="utf-8"))
    else:
        log = WORK / "D_loop.log"
        tail = log.read_text(encoding="utf-8").splitlines()[-40:] if log.exists() else []
        lines += ["**La boucle n'a pas produit de rapport.** Fin du journal :", "```", *tail, "```"]
    lines += ["", "## Durées du notebook"]
    for step, info in REPORT["steps"].items():
        extra = {k: v for k, v in info.items() if k not in ("status", "seconds")}
        lines.append(f"- {step} : {info['status']}, {info['seconds'] / 60:.1f} min {extra or ''}")
    lines.append(f"- durée totale : {(time.time() - T_START) / 60:.1f} min")
except Exception as e:
    lines.append(f"ERREUR pendant le rapport : {type(e).__name__}: {e}")

text = "\n".join(lines)
(WORK / "report.md").write_text(text, encoding="utf-8")
save_report()
print(text)

# Zip: UMA cache (old + new relaxations), MP and band-gap caches, loop outputs and logs (no checkpoint, no key).
try:
    stage = Path("/content/autolab_loop_results")
    shutil.rmtree(stage, ignore_errors=True)
    for sub in ("relax", "md", "mp", "bandgap"):
        if (CACHE / sub).exists():
            shutil.copytree(CACHE / sub, stage / "cache" / sub, ignore=shutil.ignore_patterns("entries"))
    shutil.copytree(WORK, stage / "run")
    archive = shutil.make_archive("/content/autolab_loop_results", "zip", stage)
    print(f"\nZip prêt : {archive} ({os.path.getsize(archive) / 1e6:.1f} Mo)")
    from google.colab import files
    files.download(archive)
except Exception as e:
    print(f"Zip/téléchargement : {type(e).__name__}: {e} — le fichier reste dans le panneau Fichiers de Colab.")